# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akinns247/Starter_Notebook247/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*
**Question:** Can historical search and engagement signals identify which pages most need a
content refresh, well enough to usefully rank them for a human reviewer with limited time?

**Decision this supports:** when a content reviewer can only look at a handful of pages this
week, which ones should they open first — instead of picking alphabetically or by publish date.


In [1]:
# Imports used throughout this notebook
import pandas as pd
import numpy as np
from pathlib import Path


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*
**Source:** the FlyRank ML Internship dataset — 78,835,655 daily fact rows, 104 pseudonymized
clients, 519,606 pseudonymized content items. **Time window:** March 2026, a mid-panel month
(the most recent days in this release are intentionally incomplete). **Grain:** one row = one
content item, one month. **Working sample:** 27,532 rows after dropping rows missing core
fields, split 21,884 train (24 clients) / 5,648 test (7 clients), zero client overlap.
**Excluded:** any future/post-refresh metric (would leak the outcome).


In [2]:
# Section 2 code — load & clean
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in possible_paths if p.exists()), None)
if data_path:
    df = pd.read_csv(data_path)
    repo_root = data_path.resolve().parent.parent.parent  # data/raw/file.csv -> repo root
else:
    df = pd.read_csv(
        "https://raw.githubusercontent.com/akinns247/Starter_Notebook247/main/data/raw/content_refresh_anonymized.csv"
    )
    repo_root = Path.cwd()  # running from a fresh clone at repo root, or Colab without local data

df["trend_direction"] = df["trend_direction"].astype(str).str.strip().str.lower()
df = df.dropna(subset=["client_id","ctr","avg_position","trend_direction","search_volume"]).copy()
print("Working sample:", df.shape)
print("Resolved repo root:", repo_root)


Working sample: (27532, 44)
Resolved repo root: /home/claude/Starter_Notebook247


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*
**Label (proxy, not ground truth):** `needs_refresh` = 1 if CTR ≤ 25th percentile AND trend is
declining AND avg. position ≥ 75th percentile — thresholds computed on train only, applied
unchanged to test. **Features:** 8 numeric + 3 categorical, all knowable at review time.
**Baseline:** Week-4 rule-based score (same signal family), evaluated on the identical split.
**Validation:** client-grouped 80/20 split. **Leakage check:** audited final features for
proxy-label ingredients and identifiers — none found, check passed.


In [3]:
# Section 3 code — split, label, leakage check
from sklearn.model_selection import GroupShuffleSplit

numeric_features = ["search_volume","competition","cpc","word_count","char_count","engagement_rate","scroll_rate","ai_traffic_pct"]
categorical_features = ["content_type","main_intent","competition_level"]
model_features = numeric_features + categorical_features
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

ctr_cutoff = train["ctr"].quantile(0.25)
position_cutoff = train["avg_position"].quantile(0.75)
train["needs_refresh"] = ((train["ctr"] <= ctr_cutoff) & (train["trend_direction"]=="down") & (train["avg_position"]>=position_cutoff)).astype(int)
test["needs_refresh"] = ((test["ctr"] <= ctr_cutoff) & (test["trend_direction"]=="down") & (test["avg_position"]>=position_cutoff)).astype(int)

overlap = set(train["client_id"]).intersection(set(test["client_id"]))
leak_check = [f for f in model_features if f in ("ctr","trend_direction","avg_position","client_id","content_id")]

print("Train:", len(train), "| Test:", len(test), "| Client overlap:", len(overlap))
print("Leakage audit — flagged features:", leak_check if leak_check else "none (PASS)")


Train: 21884 | Test: 5648 | Client overlap: 0
Leakage audit — flagged features: none (PASS)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*
Random Forest vs. Week-4 baseline, same held-out split (5,648 rows, 170 positive, 3.0% base rate):

| Metric | Baseline | Random Forest |
|---|---|---|
| Precision@20 | 0.00 | 0.05 |
| Precision | 0.0237 | 0.0358 |
| Recall | 0.4882 | 0.2647 |
| F1 | 0.0453 | 0.0630 |
| ROC-AUC | 0.3439 | 0.5682 |

The model wins on ranking quality but has *lower* recall than the simpler rule — a real
trade-off, not a clean win.


In [4]:
# Section 4 code — train, score, compare to baseline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

X_train, X_test = train[model_features].copy(), test[model_features].copy()
y_train, y_test = train["needs_refresh"].copy(), test["needs_refresh"].copy()

numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer([("num", numeric_transformer, numeric_features), ("cat", categorical_transformer, categorical_features)], remainder="drop")
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=5, class_weight="balanced_subsample", random_state=42, n_jobs=-1)
model = Pipeline([("preprocessor", preprocessor), ("model", rf)])
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]
preds = (model_scores >= 0.5).astype(int)

baseline_row = {"Precision@20":0.00,"Precision":0.0237,"Recall":0.4882,"F1":0.0453,"ROC-AUC":0.3439}
top20_idx = np.argsort(model_scores)[::-1][:20]
rf_row = {
    "Precision@20": round(float(y_test.values[top20_idx].sum()/20), 4),
    "Precision": round(float(precision_score(y_test, preds, zero_division=0)), 4),
    "Recall": round(float(recall_score(y_test, preds, zero_division=0)), 4),
    "F1": round(float(f1_score(y_test, preds, zero_division=0)), 4),
    "ROC-AUC": round(float(roc_auc_score(y_test, model_scores)), 4),
}
comparison = pd.DataFrame([baseline_row, rf_row], index=["Week-4 baseline","Random Forest"])
display(comparison)


,Precision@20,Precision,Recall,F1,ROC-AUC
Week-4 baseline,0.00,0.0237,0.4882,0.0453,0.3439
Random Forest,0.05,0.0358,0.2647,0.0630,0.5682


## 5. Limitations

*What this work cannot claim.*
- The label is a proxy, never independently verified by an editor.
- Absolute performance is weak: ROC-AUC 0.568 is only modestly above chance.
- Recall is a real weakness — roughly 3 in 4 proxy-positive pages are missed.
- Validated on 7 clients, 2 content types only — do not assume broader generalization.
- No causal claim: nothing here shows a refresh would actually improve a page's performance.


In [5]:
# Section 5 code — quantify the limitations named above
print("Proxy-positive base rate:", round(float(y_test.mean()), 4))
print("Recall gap vs. baseline:", round(baseline_row["Recall"] - rf_row["Recall"], 4), "(RF misses more true positives than the simpler rule)")
print("ROC-AUC above chance:", round(rf_row["ROC-AUC"] - 0.5, 4))


Proxy-positive base rate: 0.0301
Recall gap vs. baseline: 0.2235 (RF misses more true positives than the simpler rule)
ROC-AUC above chance: 0.0682


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*
Ranked, reason-coded queue for human review (same logic as ML-10):

| Reason code | Count |
|---|---|
| COMBINED_OPPORTUNITY | 2,799 |
| LOW_CTR | 859 |
| DECLINING_TREND | 814 |
| MODEL_SIGNAL_ONLY | 681 |
| HIGH_SEARCH_OPPORTUNITY | 289 |
| POOR_POSITION | 206 |

No-go: never auto-publish, auto-delete, auto-redirect, or auto-edit claims from this queue
alone. A human decides every action.


In [6]:
# Section 6 code — ranked, reason-coded queue (same logic as ML-10)
action_queue = test[["content_id","search_volume","ctr","avg_position","trend_direction","needs_refresh"]].copy()
action_queue["model_score"] = model_scores
action_queue = action_queue.sort_values("model_score", ascending=False).reset_index(drop=True)
action_queue.insert(0, "priority_rank", range(1, len(action_queue)+1))

ctr_cutoff_a = action_queue["ctr"].quantile(0.25)
search_cutoff_a = action_queue["search_volume"].quantile(0.75)
position_cutoff_a = action_queue["avg_position"].quantile(0.75)

def make_reason(row):
    reasons = []
    if str(row["trend_direction"]).lower() == "down": reasons.append("DECLINING_TREND")
    if row["ctr"] <= ctr_cutoff_a: reasons.append("LOW_CTR")
    if row["search_volume"] >= search_cutoff_a: reasons.append("HIGH_SEARCH_OPPORTUNITY")
    if row["avg_position"] >= position_cutoff_a: reasons.append("POOR_POSITION")
    if len(reasons) >= 2: return "COMBINED_OPPORTUNITY"
    if len(reasons) == 1: return reasons[0]
    return "MODEL_SIGNAL_ONLY"

action_queue["reason_code"] = action_queue.apply(make_reason, axis=1)
display(action_queue["reason_code"].value_counts().to_frame("count"))


,count
reason_code,
COMBINED_OPPORTUNITY,2799
LOW_CTR,859
DECLINING_TREND,814
MODEL_SIGNAL_ONLY,681
HIGH_SEARCH_OPPORTUNITY,289
POOR_POSITION,206


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*
Exports the three charts embedded in the deployed paper (metrics comparison, ROC curve,
precision@k) to `work/outputs/` and `docs/img/`.


In [7]:
# Section 7 code — export the paper's artifacts + the sealed metrics file
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve
from sklearn.inspection import permutation_importance
import os, json

outputs_dir = repo_root / "work" / "outputs"
img_dir = repo_root / "docs" / "img"
outputs_dir.mkdir(parents=True, exist_ok=True)
img_dir.mkdir(parents=True, exist_ok=True)

# Chart 1 — metrics comparison
metrics_names = list(baseline_row.keys())
x = np.arange(len(metrics_names)); w = 0.35
fig, ax = plt.subplots(figsize=(7,4.2))
ax.bar(x-w/2, list(baseline_row.values()), w, label="Week-4 rule baseline", color="#B9791F")
ax.bar(x+w/2, list(rf_row.values()), w, label="Random Forest", color="#0F8B8D")
ax.set_xticks(x); ax.set_xticklabels(metrics_names); ax.set_ylabel("score")
ax.set_title("Model vs. baseline — same held-out split"); ax.legend()
plt.tight_layout(); plt.savefig(img_dir / "chart_metrics_comparison.png", dpi=150); plt.close()

# Chart 2 — ROC curve
fpr, tpr, _ = roc_curve(y_test, model_scores)
fig, ax = plt.subplots(figsize=(5.5,5))
ax.plot(fpr, tpr, color="#0F8B8D", linewidth=2, label=f"Random Forest (AUC = {rf_row['ROC-AUC']:.3f})")
ax.plot([0,1],[0,1], linestyle="--", color="#9CA3AF", label="Random guess (AUC = 0.500)")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curve — Random Forest on held-out clients"); ax.legend(loc="lower right")
plt.tight_layout(); plt.savefig(img_dir / "chart_roc_curve.png", dpi=150); plt.close()

# Chart 3 — precision@k
order = np.argsort(model_scores)[::-1]
y_sorted = y_test.values[order]
ks = list(range(5, 505, 5))
precision_at_k = [y_sorted[:k].sum()/k for k in ks]
fig, ax = plt.subplots(figsize=(7,4.2))
ax.plot(ks, precision_at_k, color="#0F8B8D", linewidth=2)
ax.axhline(y_test.mean(), color="#B9791F", linestyle="--", label=f"Base rate ({y_test.mean():.3f})")
ax.set_xlabel("k (top-k ranked pages reviewed)"); ax.set_ylabel("Precision@k")
ax.set_title("Precision at k"); ax.legend()
plt.tight_layout(); plt.savefig(img_dir / "chart_precision_at_k.png", dpi=150); plt.close()

# Permutation importance (for the sealed metrics file + Interpretation section)
perm = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)

# Error counts
false_pos_n = int(((y_test==0) & (preds==1)).sum())
false_neg_n = int(((y_test==1) & (preds==0)).sum())

# THE SEALED METRICS FILE — source of truth this page/report cite
sealed_metrics = {
    "run_date": pd.Timestamp.now().strftime("%Y-%m-%d"),
    "random_state": 42,
    "n_train_rows": int(len(train)), "n_test_rows": int(len(test)),
    "n_train_clients": int(train["client_id"].nunique()), "n_test_clients": int(test["client_id"].nunique()),
    "client_overlap": int(len(overlap)),
    "base_rate": round(float(y_test.mean()), 4),
    "baseline_metrics": baseline_row,
    "random_forest_metrics": rf_row,
    "false_positives": false_pos_n,
    "false_negatives": false_neg_n,
    "permutation_importance": importance.round(4).to_dict(),
}
with open(outputs_dir / "capstone_metrics.json", "w") as f:
    json.dump(sealed_metrics, f, indent=2)

print("Exported 3 charts to", img_dir)
print("Exported sealed metrics to", outputs_dir / "capstone_metrics.json")
print(json.dumps(sealed_metrics, indent=2))


Exported 3 charts to /home/claude/Starter_Notebook247/docs/img
Exported sealed metrics to /home/claude/Starter_Notebook247/work/outputs/capstone_metrics.json
{
  "run_date": "2026-08-26",
  "random_state": 42,
  "n_train_rows": 21884,
  "n_test_rows": 5648,
  "n_train_clients": 24,
  "n_test_clients": 7,
  "client_overlap": 0,
  "base_rate": 0.0301,
  "baseline_metrics": {
    "Precision@20": 0.0,
    "Precision": 0.0237,
    "Recall": 0.4882,
    "F1": 0.0453,
    "ROC-AUC": 0.3439
  },
  "random_forest_metrics": {
    "Precision@20": 0.05,
    "Precision": 0.0358,
    "Recall": 0.2647,
    "F1": 0.063,
    "ROC-AUC": 0.5682
  },
  "false_positives": 1213,
  "false_negatives": 125,
  "permutation_importance": {
    "engagement_rate": 0.0313,
    "scroll_rate": 0.024,
    "char_count": 0.021,
    "word_count": 0.0107,
    "competition": 0.0071,
    "competition_level": 0.0041,
    "content_type": 0.0,
    "main_intent": -0.0015,
    "ai_traffic_pct": -0.0022,
    "cpc": -0.0092,
    "s

## 5-Minute Demo Outline (ML-12)

For the Week-8 showcase, if presenting:

**1. Question (30s)** — Can historical search and engagement signals identify which pages most
need a content refresh, well enough to usefully rank them for a human reviewer?

**2. Method (60s)** — A grouped-by-client Random Forest, trained on a 27,532-row working sample
from FlyRank's 78.8-million-row search and engagement warehouse. Validated on 7 clients the
model never saw during training (client-disjoint split), against a simple rule-based baseline
on the identical split.

**3. One chart (90s)** — `docs/img/chart_metrics_comparison.png`: baseline vs. Random Forest
across five metrics. Point at the trade-off directly — the model wins on ranking quality
(ROC-AUC, Precision@20) but the baseline actually catches more true positives (Recall 0.49 vs.
0.26).

**4. One honest result (60s)** — ROC-AUC of 0.568 is a real but modest lift over the baseline's
0.344 — only moderately above chance. This is a narrow, directional signal, not a reliable
classifier, and that's said plainly rather than oversold.

**5. One recommendation (60s)** — The score powers a ranked, reason-coded queue for human
review. The signal is concentrated in the first ~50 ranked pages (see the precision@k chart);
past that it's barely better than picking at random. Never auto-publish, auto-delete, or
auto-edit from this queue — a human decides every action.


## Shareable Cuts (ML-12)

**Social post** (one finding + one chart + one method sentence + link):

> Built a client-grouped Random Forest on 27.5K rows of real FlyRank search & engagement data
> to rank which pages most need a content refresh — validated on 7 completely unseen clients,
> not just random rows. It beats a simple rule on ranking quality (ROC-AUC 0.57 vs. 0.34) but
> actually has *lower* recall (0.26 vs. 0.49) — a real trade-off, reported honestly instead of
> hidden. Full paper + charts: https://akinns247.github.io/Starter_Notebook247/

**Employer 3-sentencer** (what I built · on what data · what it showed):

> I built a Random Forest model that ranks website pages by how urgently they need a content
> refresh, trained and validated on 27,532 real production rows from FlyRank's 78.8-million-row
> search and engagement warehouse. Using a client-disjoint validation split — testing on 7
> clients the model never saw — it modestly but genuinely outperformed a simple rule-based
> baseline on ranking quality (ROC-AUC 0.57 vs. 0.34), while honestly falling short on recall, a
> trade-off I documented rather than hid. The output is a ranked, reason-coded action queue
> built for human review, not automation, deployed as a public research paper with full
> reproducibility.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
